In [56]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

In [5]:
from langchain.tools import tool

In [6]:
from typing import List, Union, Optional, Dict, Literal, Callable

In [7]:
import tiktoken

In [8]:
import os
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
from pathlib import Path

In [11]:
import secrets

In [12]:
import asyncio

In [13]:
import subprocess

In [14]:
import frontmatter as fm

In [15]:
import tempfile as tf

In [43]:
from pydantic import BaseModel, Field

In [66]:
import numpy as np

In [16]:
Path.cwd()

PosixPath('/home/butcher/projects/llm_wiki/experiments')

In [17]:
with tf.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write("Hello world")
    path = f.name

print(path)

/tmp/tmp9qn625a6


In [18]:
with open(path, "r") as f:
    print(f.read())

Hello world


In [19]:
fid = path.split("/")[-1]

In [20]:
os.unlink(path)

In [21]:
key = "nt_" + secrets.token_hex(6)[:6]

In [22]:
key

'nt_45b501'

In [23]:
llm = ChatOpenAI(
    base_url= os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("LLM"),
)

In [58]:
emb = OpenAIEmbeddings(
    base_url= os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("EMB"),
)

In [24]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [25]:
client = MultiServerMCPClient(
    {
        "webgent": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "--directory",
                "/home/butcher/projects/webgent-mcp",
                "webgent",
            ],
            "env": {
                "WEBGENT_TRANSPORT": "stdio",
                "WEBGENT_HEADLESS": "false",
            },
        }
    }
)

In [26]:
class Server:
    
    def __init__(
        self,
        name: str,
        transport: Literal["stdio", "http"] = "http",
        url: Optional[str] = None,
        command: Optional[str] = None,
        args: Optional[List[str]] = None,
        env: Optional[Dict[str, Any]] = None
    ):
        
        self.name = name
        self.transport = transport
        self.url = url
        self.command = command
        self.args = args
        self.env = env

    def dump_json(self):
        config = {}
        config["transport"] = self.transport
        if self.env is not None:
            config["env"] = self.env
        
        match self.transport:
            case "http":
                config["url"] = self.url
            case "stdio":
                config["command"] = self.command
                config["args"] = self.args

        return config  

In [27]:
class DuplicateValueError(Exception):
    """Exception raised when a duplicate value is detected."""
    pass

In [28]:
class MCPClient:
    
    def __init__(
        self,
        servers: List[Server]
    ):
        self.servers = servers
        self.names = [server.name for server in servers]
        self.client = None

    def connect(self):
        servers = {}
        for server in self.servers:
            if server.name in servers:
                raise DuplicateValueError(f"server name {server.name} found twice")
            servers[server.name] = server.dump_json()

        self.client = MultiServerMCPClient(servers)

    def append(self, server: Server):
        if server.name in self.names:
            raise DuplicateValueError(f"server with name {server.name} already there")
        self.server.append(server)
        return True

    async def get_tools(self):
        if self.client is None:
            self.connect()

        return await self.client.get_tools()

In [29]:
server = Server(
    name="myserver",
    transport="http",
    url="http://127.0.0.1:8000/mcp"
)

In [30]:
client = MCPClient(
    servers=[server]
)

In [34]:
import json

In [35]:
class InternalTools:
    @classmethod
    def tools(cls):
        
        @tool("collapsed_tool_result", description="Fetch old collapsed tool result using tool call id.")
        def collapsed_tool_result(tool_call_id: str) -> str:
            # Ensure path is a Path object and join it with the filename
            file_path = Thread.get_tool_result_path() / tool_call_id
            
            try:
                # Direct, clean reading using pathlib
                return file_path.read_text(encoding="utf-8")
            except Exception as e:
                return str(e)
                
        @tool("execute_command", description="Run any terminal CLI commands")
        def execute_command(command: str) -> str:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=30,
            )
            
            return result.stdout

        @tool("init_note", description= "Initialise the note")
        def init_note() -> str:
            path = Path.cwd() / "notes"
            path.mkdir(parents=True, exist_ok=True)
            key = "nt_" + secrets.token_hex(6)[:6]
            with open(path / f"{key}.md" , "w", encoding="utf-8") as f:
                f.write("")
            return f"Initialized notes successfully use `{key}` key for access and operations."

        @tool("add_note", description="Add a note")
        def add_note(key: str, title: str, body: str) -> str:
            note_id = secrets.token_hex(6)[:6]
            note = f"<--- note_id: {note_id} title: {title} --->"
            with open(path / f"{key}.md" , "r", encoding="utf-8") as f:
                text = f.read()
                post = fm.loads(text)
                content = post.content
                
            

        return [collapsed_tool_result]


In [37]:
from langchain_openai import ChatOpenAI
from typing import List, Iterator, Any

class Agent:
    def __init__(self, model: ChatOpenAI, tools=List):
        self.model = model
        self.tools = tools
        if self.tools:
            self.tools.extend(InternalTools.tools())
            self.model = self.model.bind_tools(self.tools)
        self.tool_map = {tool.name: tool for tool in self.tools}
            
    async def ainvoke(self, thread: Thread, self_append: bool = True):
        if thread.tail is not None:
            thread = thread.tail
        thread.agent = self
        while True:
            response = await self.model.ainvoke(thread.messages)
            
            if not self_append:
                thread.agent = None
                return response
                
            thread.append(response)
            if response.tool_calls:
                for tool in response.tool_calls:
                    args=tool["args"]
                    call_id = tool["id"]
                    name = tool["name"]
                    result = await self.tool_map[name].ainvoke(args)
                    thread.append(ToolMessage(name=name, content=str(result), tool_call_id=call_id))
            else:
                thread.agent = None
                return response

    def invoke(self, thread: Thread, self_append: bool = True):
        return asyncio.run(
            self.ainvoke(
                thread,
                self_append=self_append,
            )
        )

    def parse(self, schema, message: str):
        parser = self.model.with_structured_output(schema, strict=True)
        return parser.invoke(message)
    
    def stream(
        self,
        thread: Thread,
        self_append: bool = True,
    ) -> Iterator[Any]:

        if thread.tail is not None:
            thread = thread.tail

        thread.agent = self

        try:
            while True:

                # Accumulated complete AI response
                response = None

                # Stream from model
                for chunk in self.model.stream(thread.messages):

                    # Send chunk to caller immediately
                    yield chunk

                    # Accumulate chunks
                    if response is None:
                        response = chunk
                    else:
                        response = response + chunk

                # If requested, save complete response
                if not self_append:
                    return

                thread.append(response)

                # Check for tool calls after the complete
                # streamed response has been assembled
                if response.tool_calls:

                    for tool in response.tool_calls:

                        args = tool["args"]
                        call_id = tool["id"]
                        name = tool["name"]

                        result = self.tool_map[name].invoke(args)

                        tool_message = ToolMessage(
                            name=name,
                            content=result,
                            tool_call_id=call_id,
                        )

                        thread.append(tool_message)

                        # Continue while-loop.
                        # The next iteration sends:
                        #
                        # previous messages
                        # + AIMessage(tool_call)
                        # + ToolMessage(result)
                        #
                        # back to the model.

                else:
                    return

        finally:
            thread.agent = None


    def __ror__(self, thread: Thread):
        return self.invoke(thread)

In [38]:
class ThreadHideRule:
    def __init__(
        self,
        name: str,
        message: str,
    ):
        self.name = name
        self.message = message

In [39]:
class AutoToolHideRule:
    def __init__(
        self,
        token_limit: int,
        per_tool_token_limit: Optional[int] = None
    ):
        self.token_limit = token_limit
        self.per_tool_token_limit = per_tool_token_limit

In [40]:
class Thread:
    def __init__(
        self, 
        messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]] = None, 
        system_prompt: Union[str, SystemMessage] = None,
        compression_prompt: str = None,
        token_limit: int = None,
        tool_hide_rules: List[Union[ThreadHideRule, AutoToolHideRule]] = None   
    ):
        self.messages = []
        self.system_prompt = system_prompt
        self.compression_prompt = compression_prompt
        self.token_limit = token_limit
        self.agent = None
        self.encoder = tiktoken.encoding_for_model("gpt-4o-mini")
        self.root = None
        self.parent = None
        self.child = None
        self.tail = None
        self.tool_hide_rules = tool_hide_rules
        self.path = Path.cwd() / "tool_results"
        self.path.mkdir(parents=True, exist_ok=True)

        
        if messages is not None:
            index = self._find_system_message(messages)
            if index == -1 or index == 0:
                self.messages = messages
            else:
                raise Valueerror(
                    f"system message not at the starting, it was found at {index} index"
                )
                    
        if self.system_prompt is not None:
            index = self._find_system_message(self.messages)
            if isinstance(self.system_prompt, str):
                self.system_prompt = SystemMessage(self.system_prompt)
            if index == 0:
                if len(self.messages) == 0:
                    self.append(self.system_prompt)
                else:
                    self[0] = self.system_prompt
            elif index == -1:
                self.messages = [self.system_prompt] + self.messages
            else:
                pass

    @classmethod
    def get_tool_result_path(cls):
        path = Path.cwd() / "tool_results"
        path.mkdir(parents=True, exist_ok=True)
        return path

    def _find_system_message(self, messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]]):
        for i in range(len(messages)):
            if isinstance(messages[i], SystemMessage):
                return i
        return -1

    def count_token(self):
        if self.tail is not None:
            content = [m.content for m in self.tail]
        else:
            content = [m.content for m in self]
        merged = "\n".join(content)
        return len(self.encoder.encode(merged))

    def calculate_tokens(self, content):
        return len(self.encoder.encode(content))
        
        
    def append(self, message: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.root is not None:
            tool_hide_rules = self.root.tool_hide_rules
        else:
            tool_hide_rules = self.tool_hide_rules

        thread_hide_rules = [rule for rule in tool_hide_rules if isinstance(rule, ThreadHideRule)]
        
        auto_tool_hide_rules = None
        for rule in tool_hide_rules:
            if isinstance(rule, AutoToolHideRule):
                auto_tool_hide_rules = rule
                break
        
            
        if isinstance(message, ToolMessage) and tool_hide_rules is not None:
            name = message.name
            match = False
            tool_hide_rule = None
            for rule in thread_hide_rules:
                if rule.name == name:
                    match = True
                    tool_hide_rule = rule
                    break
            if match:
                for m in reversed(self):
                    if isinstance(m, ToolMessage) and m.name == name:
                        self.save_tool_result(m)
                        m.content = tool_hide_rule.message + f"\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                        break
                        
        if auto_tool_hide_rules is not None:
            total_tokens = self.count_token()
            if total_tokens >= auto_tool_hide_rules.token_limit:
                if auto_tool_hide_rules.per_tool_token_limit is not None:
                    per_tool_token_limit = auto_tool_hide_rules.per_tool_token_limit
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                else:
                    per_tool_token_limit = 8_000
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"

            
        

                        
        token_usuage = self.count_token()
        if self.token_limit is not None and token_usuage > self.token_limit and self.agent is not None and self.compression_prompt is not None:
            self.messages.append(HumanMessage(self.compression_prompt))
            compression_report = self.agent.invoke(self, self_append=False).content
            self.messages.pop()
            
            new_thread = self.copy()
            self.child = new_thread
            new_thread.parent = self
            new_thread.root = self.root if self.root is not None else self
            self.root.tail = new_thread

            new_thread.messages = []
            if isinstance(self.root[0], SystemMessage):
                new_thread.append(self.root[0])
            new_thread.append(HumanMessage(compression_report)) 
        else:
            self.messages.append(message)

    def save_tool_result(self, message: ToolMessage) -> bool:
        try:
            with open(self.path / str(message.tool_call_id), "w", encoding="utf-8") as f:
                f.write(message.content)
            return True
        except:
            return False

    def count(self):
        counts = {
            "depth":0,
            "system":0, 
            "human":0,
            "ai":0,
            "tool":0
        }
        thread = self
        depth = 0
        if isinstance(thread[0], SystemMessage):
            counts["system"]=1
        while True:
            for m in thread:
                if isinstance(m, AIMessage):
                    counts["ai"]+=1
                elif isinstance(m, HumanMessage):
                    counts["human"]+=1
                elif isinstance(m, ToolMessage):
                    counts["tool"]+=1
                else:
                    pass
            if thread.child is None:
                break
            else:
                thread = thread.child
                depth += 1
        counts["depth"] = depth
        return counts

    def __ror__(self, other: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.tail is not None:
            self.tail.append(other)
        else:
            self.append(other)

    # def __add__(self, other: Thread):
    #     new_thread = Thread()
    #     new_thread.messages = self.messages + other.messages
    #     return new_thread

    def __str__(self):
        counts = self.count()
        return json.dumps(counts)

    def __repr__(self):
        counts = self.count()
        return json.dumps(counts)

    def __iter__(self):
        if self.tail is not None:
            for msg in self.tail.messages:
                yield msg
        else:
            for msg in self.messages:
                yield msg
                
    def __getitem__(self, index):
        if self.tail is not None:
            return self.tail.messages[index]
        else:
            return self.messages[index]

    def __len__(self):
        if self.tail is not None:
            return len(self.tail.messages)
        else:
            return len(self.messages)

    def __setitem__(self, index, value):
        if self.tail is not None:
            self.tail.messages[index] = value
        else:
            self.messages[index] = value

    def __copy__(self):
        new_instance = Thread()
        return new_instance
        

In [41]:
myagent = Agent(
    model=llm,
    tools=[calculator]
)

In [49]:
class User(BaseModel):
    name: str = Field(..., description= "Name of the User")
    age: int = Field(..., description= "Age of the User")

class UserDetails(BaseModel):
    details: List[User] = Field(..., description= "List of user details")

In [132]:
class Vector:
    def __init__(self, model: OpenAIEmbeddings, dim: Union[int, bool] = False):
        self.model = model
        self.dim = dim

    def _reduce_and_normalize(self, arr: np.ndarray):
        arr = arr[:self.dim]
        norms = np.linalg.norm(arr, keepdims=True)
        norms[norms == 0] = 1
        return arr / norms

    def embed(self, doc: Union[str, List[str]]):
        if isinstance(doc, str):
            emb = np.array(self.model.embed_query(doc))
            print(len(emb))
            if self.dim and self.dim <= len(emb):
                print("Reducing.....")
                return self._reduce_and_normalize(emb)
            return emb
            
        elif isinstance(doc, list):
            emb = np.array(self.model.embed_documents(doc))
            if self.dim and self.dim <= len(emb[0]):
                arr = []
                for e in emb:
                    arr.append(self._reduce_and_normalize(e))
                return np.array(arr)
            return emb

In [180]:
v = Vector(model = emb, dim = 256)

In [181]:
e = v.embed([message, message[:20]])

In [182]:
q = v.embed("Provide the user details")

4096
Reducing.....


In [183]:
len(e)

2

In [184]:
e

array([[ 0.12376917,  0.06537104, -0.06057717, -0.1002356 , -0.02898116,
        -0.07147234, -0.03333923, -0.04663134,  0.01361897,  0.03944053,
        -0.01356449, -0.04575973,  0.16734987,  0.01721437,  0.00667329,
         0.03355713,  0.02451414,  0.02091873,  0.04946409,  0.06362781,
        -0.031596  ,  0.11330981,  0.17868085, -0.08367493, -0.11156658,
         0.07975267,  0.01198469,  0.11505303,  0.02985278, -0.03007068,
        -0.07583041,  0.04663134,  0.10938754,  0.05098941, -0.12464079,
        -0.00882509, -0.11679625, -0.02658422,  0.0039495 , -0.03224971,
        -0.11330981, -0.00446702, -0.02484099, -0.04663134,  0.03399294,
         0.05752652, -0.09064785, -0.03007068,  0.03355713, -0.07583041,
        -0.02516785, -0.01775913, -0.01296526,  0.01427268,  0.1194111 ,
         0.00675501, -0.06972911,  0.01089517, -0.11853949, -0.07147234,
        -0.17170793, -0.02048293,  0.04183747, -0.04684924, -0.02266196,
        -0.02549471, -0.01449058,  0.01356449,  0.0

In [185]:
q@e.T

array([0.60159707, 0.62242315])

In [148]:
e0 = e[0]
e1 = e[1]

In [149]:
e0@e1

np.float64(0.6352632498040972)

In [137]:
sum(e)

np.float64(-0.5296336237043479)

In [92]:
dim = 64

In [76]:
e = e[:dim]

In [78]:
e 

array([ 0.0334847 ,  0.01757353, -0.01674235, -0.02707274, -0.00789622,
       -0.01959211, -0.00914299, -0.01306141,  0.00369579,  0.01080535,
       -0.00353252, -0.01246771,  0.04535871,  0.00489803,  0.00184789,
        0.00896488,  0.0065307 ,  0.00549173,  0.01383322,  0.01709857,
       -0.00878677,  0.03087242,  0.04892091, -0.0227981 , -0.03027872,
        0.0216107 ,  0.00338409,  0.03134738,  0.00801496, -0.00837118,
       -0.02066077,  0.01246771,  0.02980376,  0.01371448, -0.03419714,
       -0.00255291, -0.03182234, -0.00721346,  0.00107608, -0.00908362,
       -0.0311099 , -0.00138777, -0.00664944, -0.01240834,  0.00938047,
        0.01555495, -0.02446046, -0.00843055,  0.00902425, -0.02054203,
       -0.00700566, -0.00463086, -0.00316145,  0.00400748,  0.03277226,
        0.00186274, -0.01923589,  0.00304271, -0.0322973 , -0.01982959,
       -0.04654611, -0.00561047,  0.01145842, -0.01264582])

In [79]:
norms = np.linalg.norm(e, keepdims=True)

In [80]:
norms[norms == 0] = 1
e = e / norms

In [81]:
e

array([ 0.21283461,  0.11170043, -0.1064173 , -0.17207904, -0.05018972,
       -0.12453088, -0.05811441, -0.08302059,  0.02349105,  0.06868067,
       -0.0224533 , -0.07924692,  0.28830788,  0.03113272,  0.01174553,
        0.05698232,  0.04151029,  0.03490638,  0.08792635,  0.1086815 ,
       -0.05585021,  0.19623048,  0.31094984, -0.14490867, -0.19245682,
        0.13736134,  0.02150988,  0.19924942,  0.05094445, -0.05320865,
       -0.13132348,  0.07924692,  0.18943789,  0.08717162, -0.217363  ,
       -0.01622675, -0.20226834, -0.04585001,  0.00683976, -0.05773705,
       -0.19773994, -0.00882094, -0.04226503, -0.07886956,  0.05962388,
        0.09886997, -0.15547492, -0.05358602,  0.05735968, -0.13056875,
       -0.04452923, -0.02943457, -0.02009476,  0.02547223,  0.2083062 ,
        0.01183987, -0.12226669,  0.01934002, -0.20528727, -0.12604035,
       -0.29585519, -0.03566112,  0.0728317 , -0.08037903])

In [54]:
message = f"""
Here are the user details : 
Manav, 25 
Shivangi, 24

return the user details in the structured json schema:
{UserDetails.model_json_schema()}
"""

In [55]:
UserDetails.model_json_schema()

{'$defs': {'User': {'properties': {'name': {'description': 'Name of the User',
     'title': 'Name',
     'type': 'string'},
    'age': {'description': 'Age of the User',
     'title': 'Age',
     'type': 'integer'}},
   'required': ['name', 'age'],
   'title': 'User',
   'type': 'object'}},
 'properties': {'details': {'description': 'List of user details',
   'items': {'$ref': '#/$defs/User'},
   'title': 'Details',
   'type': 'array'}},
 'required': ['details'],
 'title': 'UserDetails',
 'type': 'object'}

In [51]:
r = myagent.parse(schema = UserDetails, message = message)

In [186]:
r

UserDetails(details=[User(name='Manav', age=25), User(name='Shivangi', age=24)])

In [53]:
r.details

[User(name='Manav', age=25), User(name='Shivangi', age=24)]

In [42]:
thread = Thread(
    tool_hide_rules=[
         AutoToolHideRule(
             token_limit = 20_000
         )
    ]
)

In [41]:
thread.append(SystemMessage("""
    You are a 
"""))

In [46]:
HumanMessage("go to amazon") | thread

In [47]:
thread.messages

[SystemMessage(content='You are a web agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='open browser session', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 5442, 'total_tokens': 5526, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-99927e77e9ceb109', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06620-a585-70d1-9379-f0783409a4f7-0', tool_calls=[{'name': 'create_session', 'args': {'profile_name': 'default', 'initial_url': 'about:blank'}, 'id': 'chatcmpl-tool-9b1ecceeb5b49af8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5442, 'output_tokens': 84, 'total_tokens': 5526, 'input_token_details': {}, 'output_token_details': {}}),
 ToolM

In [48]:
# for chunk in myagent.stream(thread):
#     print(chunk.content, end="", flush=True)

In [49]:
response = await myagent.ainvoke(thread)

In [37]:
async with client.session("webgent") as session:
    mcp_tools = await load_mcp_tools(session)

    myagent = Agent(
        model=llm,
        tools=mcp_tools
    )

    response = await myagent.ainvoke(thread)

In [37]:
response

AIMessage(content="Hello again! 😊\n\nIs there anything you'd like me to help you with on the web today? Maybe:\n\n- Browse a website or search for something?\n- Fill out a form or log into a service?\n- Scrape some data from a page?\n- Test something on a site?\n\nJust let me know what you need and I'll get right to it!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 5487, 'total_tokens': 5602, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-947a944e2629dc75', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a065f9-35b2-77f3-98b0-97c2f8227727-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5487, 'output_tokens': 115, 'total_tokens': 5602, 'input_token_details': {}, 'output_token_details': {}})

In [31]:
response.content

'Hi there! How can I help you today? If you need me to browse a website, fill out a form, or do anything else on the web, just let me know!'

In [29]:
thread.messages

[SystemMessage(content='You are a web agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='go to amazon.in', additional_kwargs={}, response_metadata={})]

In [39]:
for m in response:
    if isinstance(m, SystemMessage):
        print("system message:\n")
    elif isinstance(m, HumanMessage):
        print("human message:\n")
    elif isinstance(m, AIMessage):
        print("ai message:\n")
    elif isinstance(m, ToolMessage):
        print("tool message:\n")
    else:
        print("other type:\n")
    print(m.model_dump_json())
    print("\n\n")

system message:

{"content":"You are a Mathematical Expression solver","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}



human message:

{"content":"Solve this particular expression : 3*4+77 use calculator tool","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null}



ai message:

{"content":"","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":70,"prompt_tokens":297,"total_tokens":367,"completion_tokens_details":null,"prompt_tokens_details":null},"model_provider":"openai","model_name":"Deepseek-vapt","system_fingerprint":"vllm-0.25.0-tp2-ep-d4f8ac0c","id":"chatcmpl-bc18df2e93c1cfe1","finish_reason":"tool_calls","logprobs":null},"type":"ai","name":null,"id":"lc_run--01a03256-a5d0-7433-8fa3-9299a904fa75-0","tool_calls":[{"name":"calculator","args":{"expression":"3*4+77"},"id":"chatcmpl-tool-8e2f16389057886f","type":"tool_call"}],"invalid_tool_calls":[],"usage_metadata":{"input_t

In [19]:
s = SystemMessage("hii")

In [20]:
s.model_dump_json()

'{"content":"hii","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}'

In [45]:
response["messages"][-1].content

'The result of the expression \\(3 \\times 4 + 77\\) is **89**.'

In [47]:
response["messages"].append(HumanMessage("Now also calculate this one 456%100"))

In [54]:
nr = agent.invoke(nr)

In [74]:
tool = mcp_tools[0]

print(tool.name)
print(tool.args_schema)

create_session
{'additionalProperties': False, 'properties': {'profile_name': {'default': 'default', 'type': 'string'}, 'initial_url': {'default': 'about:blank', 'type': 'string'}}, 'type': 'object'}


In [75]:
result = await tool.ainvoke({
    "profile_name": "default",
    "initial_url": "about:blank",
})

print(result)

[{'type': 'text', 'text': '{"session_id":"ba7b1c0e-7c32-4842-a75e-f8f91e6d5635","profile_name":"default","page_id":"b500c32e-d571-417d-bc59-821bc5cdb5a7","url":"about:blank","title":""}', 'id': 'lc_12b1d9f4-209b-4c4b-8ad2-bb751f60144e'}]


In [76]:
type(result)

list